# Complete RAG Workflows

## Learning Objectives

By completing this notebook, you will:
1. Build end-to-end RAG pipelines (Retrieval + Generation)
2. Optimize retrieval parameters (top_k, filters, reranking)
3. Design effective RAG prompts with context
4. Implement citation and source tracking
5. Evaluate RAG output quality

## Prerequisites

- Notebook 01: Knowledge Bank Creation completed
- Module 1: Prompt design completed
- Understanding of LLM context windows

## Estimated Time: 60 minutes

---

In [ ]:
learning_objectives(["Build end-to-end RAG pipelines (Retrieval + Generation)", "Optimize retrieval parameters (top_k, filters, reranking)", "Design effective RAG prompts with context", "Implement citation and source tracking", "Evaluate RAG output quality"])

## What is RAG?

**Retrieval-Augmented Generation (RAG)** combines retrieval and generation:

```
User Query → Retrieve Relevant Docs → Generate Answer with Context → Cite Sources
```

### RAG vs Fine-Tuning

| Approach | When to Use | Pros | Cons |
|----------|-------------|------|------|
| **RAG** | Facts change often, need citations | Always current, explainable | Retrieval overhead |
| **Fine-tuning** | Behavior/style changes | No retrieval needed | Expensive updates |

### Key Insight

**RAG is essential when**:
- Information updates frequently (market data, news)
- You need source attribution (compliance, trust)
- Data is too large for context window
- Multiple knowledge sources must be combined

## Setup

Import libraries and load the Knowledge Bank from Notebook 01.

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
# Apply course styling
apply_course_theme()
apply_plot_theme()

In [ ]:
import dataiku
from dataiku.knowledge_bank import KnowledgeBank
from dataiku.llm import LLM
import json
from typing import List, Dict
import pandas as pd
from datetime import datetime

In [ ]:
# Recreate sample data and KB from Notebook 01
sample_reports = [
    {
        'id': 'eia_wpsr_2024_03_15',
        'title': 'EIA Weekly Petroleum Status Report - March 15, 2024',
        'date': '2024-03-15',
        'source': 'EIA',
        'report_type': 'inventory',
        'content': '''U.S. commercial crude oil inventories decreased by 1.5 million barrels. At 445.1 million barrels, 
        U.S. crude oil inventories are about 3% below the five-year average. Total motor gasoline inventories increased 
        by 0.6 million barrels. Distillate fuel inventories decreased by 0.9 million barrels and are about 6% below the 
        five-year average. Refinery utilization rate of 89.2%.'''
    },
    {
        'id': 'opec_omr_2024_03',
        'title': 'OPEC Monthly Oil Market Report - March 2024',
        'date': '2024-03-12',
        'source': 'OPEC',
        'report_type': 'market_outlook',
        'content': '''OPEC maintained its forecast for global oil demand growth in 2024 at 2.25 million barrels per day. 
        Non-OPEC supply growth projected at 1.5 million bpd, led by U.S. shale and Canadian oil sands. 
        OPEC+ production cuts of 2.2 million bpd remain in effect through Q2 2024.'''
    },
    {
        'id': 'iea_omr_2024_03',
        'title': 'IEA Oil Market Report - March 2024',
        'date': '2024-03-14',
        'source': 'IEA',
        'report_type': 'market_outlook',
        'content': '''The IEA revised 2024 global oil demand growth forecast down to 1.3 million barrels per day. 
        Global oil supply increased by 500,000 bpd month-on-month, driven by U.S., Brazil, and Guyana. 
        The IEA projects surplus of approximately 1 million bpd in H1 2024.'''
    }
]

# Simple KB implementation
class SimpleKB:
    def __init__(self, documents):
        self.documents = documents
    
    def search(self, query: str, top_k: int = 3, filters: Dict = None) -> List[Dict]:
        query_lower = query.lower()
        results = []
        
        for doc in self.documents:
            if filters:
                if not all(doc.get(k) == v for k, v in filters.items()):
                    continue
            
            score = sum(word in doc['content'].lower() for word in query_lower.split())
            if score > 0:
                results.append({
                    'text': doc['content'],
                    'score': score / len(query_lower.split()),
                    'metadata': {
                        'title': doc['title'],
                        'date': doc['date'],
                        'source': doc['source'],
                        'report_type': doc['report_type']
                    }
                })
        
        results.sort(key=lambda x: x['score'], reverse=True)
        return results[:top_k]

kb = SimpleKB(sample_reports)
llm = LLM('claude-sonnet')  # Update with your connection

print("✓ Knowledge Bank loaded with", len(sample_reports), "documents")
print("✓ LLM connection ready")

## Exercise 1: Basic RAG Pipeline

A basic RAG pipeline has three steps:
1. **Retrieve**: Get relevant documents from KB
2. **Format**: Create prompt with context
3. **Generate**: Get LLM response

In [ ]:
def basic_rag(query: str, kb: SimpleKB, llm: LLM, top_k: int = 3) -> Dict:
    """
    Basic RAG pipeline.
    
    Parameters
    ----------
    query : str
        User's question
    kb : SimpleKB
        Knowledge Bank to search
    llm : LLM
        LLM for generation
    top_k : int
        Number of documents to retrieve
    
    Returns
    -------
    dict
        answer: Generated text
        sources: Retrieved documents
        context_used: Context sent to LLM
    """
    # Step 1: Retrieve
    retrieved_docs = kb.search(query, top_k=top_k)
    
    # Step 2: Format context
    context = "\n\n---\n\n".join([
        f"Source: {doc['metadata']['title']}\n"
        f"Date: {doc['metadata']['date']}\n"
        f"Content: {doc['text']}"
        for doc in retrieved_docs
    ])
    
    # Step 3: Create prompt
    prompt = f"""Answer the question based on the provided context. If the context doesn't 
contain enough information, say so clearly. Cite specific sources when possible.

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""
    
    # Step 4: Generate
    response = llm.complete(prompt, max_tokens=300, temperature=0.3)
    
    return {
        'answer': response.text,
        'sources': retrieved_docs,
        'context_used': context,
        'tokens': response.usage.total_tokens
    }

# Test basic RAG
test_query = "What is the current state of crude oil inventories?"

result = basic_rag(test_query, kb, llm)

print("="*60)
print(f"QUESTION: {test_query}")
print("="*60)
print(f"\nANSWER:\n{result['answer']}")
print(f"\nSOURCES USED: {len(result['sources'])}")
for i, source in enumerate(result['sources'], 1):
    print(f"  {i}. {source['metadata']['source']} - {source['metadata']['date']}")
print(f"\nTOKENS: {result['tokens']}")

### Exercise 1.1: Improve the RAG Prompt

**Task**: The basic prompt is functional but could be better. Improve it to:
- Request structured output (JSON or markdown)
- Require explicit source citations
- Handle cases where information is incomplete

In [ ]:
def improved_rag(query: str, kb: SimpleKB, llm: LLM, top_k: int = 3) -> Dict:
    """
    RAG with improved prompt engineering.
    
    YOUR TASK: Modify the prompt template to:
    - Return structured JSON with fields: answer, confidence, sources_cited
    - Explicitly cite sources by name and date
    - Indicate confidence level (high/medium/low)
    - State if information is insufficient
    """
    # Retrieve
    retrieved_docs = kb.search(query, top_k=top_k)
    
    # Format context
    context = "\n\n".join([
        f"[{i+1}] {doc['metadata']['title']} ({doc['metadata']['date']})\n{doc['text']}"
        for i, doc in enumerate(retrieved_docs)
    ])
    
    # YOUR CODE HERE: Improve this prompt
    prompt = f"""
    
    """
    
    response = llm.complete(prompt, max_tokens=400, temperature=0.2)
    
    # Parse JSON response
    try:
        parsed = json.loads(response.text)
    except:
        parsed = {'answer': response.text, 'confidence': 'unknown', 'sources_cited': []}
    
    return {
        **parsed,
        'retrieved_sources': retrieved_docs,
        'tokens': response.usage.total_tokens
    }

# Test improved RAG
queries = [
    "What is OPEC's oil demand forecast for 2024?",
    "How do IEA and OPEC forecasts compare?",
    "What was the weather like in Texas last week?"  # Should indicate insufficient info
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"Q: {query}")
    print('='*60)
    
    result = improved_rag(query, kb, llm)
    
    print(f"Answer: {result.get('answer', result)}")
    print(f"Confidence: {result.get('confidence', 'N/A')}")
    if 'sources_cited' in result:
        print(f"Sources cited: {result['sources_cited']}")

# Auto-graded test
test_result = improved_rag("What is OPEC's forecast?", kb, llm)
assert 'answer' in test_result or 'error' not in str(test_result).lower(), "Should return valid answer"
print("\n✓ Exercise 1.1 passed!")

## Exercise 2: Optimizing Retrieval Parameters

Retrieval quality determines RAG quality. Key parameters:

### Top-K Selection
- **Too few** (k=1-2): Miss relevant info
- **Too many** (k>10): Noise, context bloat, higher cost
- **Sweet spot** (k=3-5): Usually optimal

### Similarity Threshold
- Only return chunks above similarity threshold
- Prevents irrelevant results
- Threshold depends on use case (0.7-0.8 typical)

In [ ]:
def compare_top_k(query: str, kb: SimpleKB, llm: LLM, k_values: List[int]) -> pd.DataFrame:
    """
    Compare RAG performance with different top_k values.
    """
    results = []
    
    for k in k_values:
        result = basic_rag(query, kb, llm, top_k=k)
        
        results.append({
            'top_k': k,
            'num_sources': len(result['sources']),
            'tokens_used': result['tokens'],
            'answer_length': len(result['answer']),
            'context_length': len(result['context_used'])
        })
    
    return pd.DataFrame(results)

# Compare top_k values
query = "What are the oil demand forecasts?"
comparison = compare_top_k(query, kb, llm, k_values=[1, 2, 3, 5])

print("Top-K Comparison:")
print(comparison.to_string(index=False))

print("\n📊 Insight:")
print(f"- k=1: Fastest but may miss important context")
print(f"- k=3: Good balance for most queries")
print(f"- k=5: More comprehensive but higher token cost")

### Exercise 2.1: Adaptive Top-K

**Task**: Implement adaptive top-k that adjusts based on:
- Query complexity (simple vs complex questions)
- Initial retrieval scores (if top results are very relevant, k=2 may suffice)
- Query type (factual lookup vs comparative analysis)

In [ ]:
def adaptive_top_k(query: str, initial_results: List[Dict]) -> int:
    """
    Determine optimal top_k based on query and initial results.
    
    YOUR TASK:
    - Analyze query complexity (word count, question words, etc.)
    - Check score distribution of initial results
    - Return appropriate k value (1-5)
    
    Parameters
    ----------
    query : str
        User's question
    initial_results : List[Dict]
        First 5 retrieved results with scores
    
    Returns
    -------
    int
        Optimal k value
    """
    # YOUR CODE HERE
    k = 3  # Default
    
    # Analyze query complexity
    query_lower = query.lower()
    
    # Check for comparison keywords
    if any(word in query_lower for word in ['compare', 'difference', 'versus', 'vs']):
        k = 5  # Need more sources for comparison
    
    # Check for simple lookup
    elif any(query_lower.startswith(word) for word in ['what is', 'who is', 'when']):
        k = 2  # Simple lookup
    
    # Check score distribution
    if len(initial_results) > 0:
        top_score = initial_results[0]['score']
        if len(initial_results) > 1:
            second_score = initial_results[1]['score']
            # If top result much better than second, k=1 may suffice
            if top_score > 2 * second_score:
                k = min(k, 2)
    
    return k

def adaptive_rag(query: str, kb: SimpleKB, llm: LLM) -> Dict:
    """
    RAG with adaptive top-k selection.
    """
    # Get initial results (max possible)
    initial_results = kb.search(query, top_k=5)
    
    # Determine optimal k
    optimal_k = adaptive_top_k(query, initial_results)
    
    print(f"Adaptive k selected: {optimal_k}")
    
    # Use only top k results
    retrieved_docs = initial_results[:optimal_k]
    
    # Continue with standard RAG
    context = "\n\n".join([doc['text'] for doc in retrieved_docs])
    
    prompt = f"""Answer based on context:

{context}

Question: {query}
Answer:"""
    
    response = llm.complete(prompt, max_tokens=200)
    
    return {
        'answer': response.text,
        'k_used': optimal_k,
        'sources': retrieved_docs
    }

# Test adaptive RAG
test_queries = [
    "What is OPEC?",  # Simple - should use k=2
    "Compare IEA and OPEC demand forecasts",  # Complex - should use k=5
    "What was the inventory change?"  # Medium - should use k=3
]

for query in test_queries:
    print(f"\nQuery: {query}")
    result = adaptive_rag(query, kb, llm)
    print(f"K used: {result['k_used']}")
    print(f"Answer: {result['answer'][:100]}...")

# Auto-graded test
assert adaptive_top_k("Compare X and Y", []) >= 3, "Comparison queries should use higher k"
assert adaptive_top_k("What is X?", []) <= 3, "Simple queries should use lower k"
print("\n✓ Exercise 2.1 passed!")

## Exercise 3: Source Citation and Attribution

**Why citations matter**:
- Trust and transparency
- Compliance and auditability
- User can verify information
- Detect hallucinations (claims without sources)

### Citation Strategies

1. **Inline citations**: "According to [EIA Report, 2024-03-15]..."
2. **Footnotes**: "...inventories fell [1]" with sources listed
3. **Source list**: Separate answer and sources section

In [ ]:
def rag_with_citations(query: str, kb: SimpleKB, llm: LLM) -> Dict:
    """
    RAG that requires inline citations.
    """
    # Retrieve
    docs = kb.search(query, top_k=3)
    
    # Format with numbered sources
    context = ""
    for i, doc in enumerate(docs, 1):
        context += f"\n[{i}] {doc['metadata']['source']} ({doc['metadata']['date']}): {doc['text']}\n"
    
    # Prompt that requires citations
    prompt = f"""Answer the question using the provided sources. 
IMPORTANT: Cite sources inline using [number] notation.

Sources:
{context}

Question: {query}

Answer (with inline citations):"""
    
    response = llm.complete(prompt, max_tokens=300, temperature=0.2)
    
    # Extract citations
    import re
    citations = re.findall(r'\[(\d+)\]', response.text)
    cited_sources = [docs[int(c)-1]['metadata'] for c in citations if int(c) <= len(docs)]
    
    return {
        'answer': response.text,
        'citations': citations,
        'cited_sources': cited_sources,
        'all_sources': [d['metadata'] for d in docs]
    }

# Test
result = rag_with_citations(
    "What are the different oil demand forecasts for 2024?",
    kb, llm
)

print("ANSWER WITH CITATIONS:")
print(result['answer'])
print(f"\nCitations found: {result['citations']}")
print(f"\nSources cited:")
for source in result['cited_sources']:
    print(f"  - {source['source']}: {source['title']}")

### Exercise 3.1: Citation Validation

**Task**: Implement validation to detect hallucinated citations.

Check that:
- All citations reference provided sources
- Claims in answer are actually in the cited source
- No unsupported claims

In [ ]:
def validate_citations(answer: str, sources: List[Dict]) -> Dict:
    """
    Validate that all citations are legitimate.
    
    YOUR TASK:
    - Extract all citation numbers from answer
    - Check each citation number is valid (1 to len(sources))
    - Flag if answer contains unsourced factual claims
    
    Returns
    -------
    dict
        valid_citations: list of valid citation numbers
        invalid_citations: list of invalid citation numbers
        unsourced_claims: potential claims without citations
        validation_score: 0-1 score
    """
    import re
    
    # YOUR CODE HERE
    citations = re.findall(r'\[(\d+)\]', answer)
    
    valid = []
    invalid = []
    
    for cite in citations:
        cite_num = int(cite)
        if 1 <= cite_num <= len(sources):
            valid.append(cite_num)
        else:
            invalid.append(cite_num)
    
    # Check for unsourced claims (sentences with numbers but no citation)
    sentences = answer.split('.')
    unsourced = []
    for sent in sentences:
        # If sentence has a number/statistic but no citation
        has_number = any(c.isdigit() for c in sent)
        has_citation = '[' in sent
        if has_number and not has_citation and len(sent.strip()) > 20:
            unsourced.append(sent.strip()[:100])
    
    # Calculate score
    total_citations = len(citations)
    score = len(valid) / total_citations if total_citations > 0 else 1.0
    score -= len(unsourced) * 0.1  # Penalty for unsourced claims
    score = max(0, min(1, score))
    
    return {
        'valid_citations': valid,
        'invalid_citations': invalid,
        'unsourced_claims': unsourced,
        'validation_score': round(score, 2)
    }

# Test validation
test_answer = """OPEC forecasts 2.25 million bpd demand growth [1], while IEA projects 
only 1.3 million bpd [2]. The difference reflects varying economic assumptions. 
Some analysts expect 2.0 million bpd growth."""  # Last sentence unsourced

test_sources = [
    {'source': 'OPEC', 'title': 'OPEC Report'},
    {'source': 'IEA', 'title': 'IEA Report'}
]

validation = validate_citations(test_answer, test_sources)

print("Citation Validation:")
print(json.dumps(validation, indent=2))

# Auto-graded test
assert 'validation_score' in validation, "Must return validation score"
assert validation['validation_score'] <= 1.0, "Score should be ≤ 1.0"
assert len(validation['valid_citations']) > 0, "Should detect valid citations"

print("\n✓ Exercise 3.1 passed!")

## Exercise 4: RAG Evaluation

Measure RAG quality across multiple dimensions:

### Evaluation Metrics

1. **Retrieval Quality**
   - Precision: Are retrieved docs relevant?
   - Recall: Did we get all relevant docs?

2. **Answer Quality**
   - Correctness: Is the answer accurate?
   - Completeness: Does it address the full question?
   - Conciseness: Is it appropriately detailed?

3. **Citation Quality**
   - Are sources cited?
   - Are citations accurate?
   - No hallucinated sources?

In [ ]:
def evaluate_rag(query: str, answer: str, sources: List[Dict], ground_truth: str = None) -> Dict:
    """
    Comprehensive RAG evaluation.
    
    YOUR TASK: Implement evaluation metrics
    """
    import re
    
    metrics = {}
    
    # 1. Citation metrics
    citations = re.findall(r'\[(\d+)\]', answer)
    metrics['num_citations'] = len(citations)
    metrics['cites_sources'] = len(citations) > 0
    
    # 2. Answer completeness (simple heuristics)
    metrics['answer_length'] = len(answer)
    metrics['addresses_question'] = any(
        word.lower() in answer.lower() 
        for word in query.split() 
        if len(word) > 4
    )
    
    # 3. Source relevance
    query_terms = set(query.lower().split())
    source_relevance = []
    for source in sources:
        source_terms = set(source['text'].lower().split())
        overlap = len(query_terms & source_terms)
        source_relevance.append(overlap / len(query_terms) if query_terms else 0)
    
    metrics['avg_source_relevance'] = sum(source_relevance) / len(source_relevance) if source_relevance else 0
    
    # 4. Overall quality score (simple weighted average)
    quality_score = (
        int(metrics['cites_sources']) * 0.3 +
        int(metrics['addresses_question']) * 0.3 +
        min(metrics['avg_source_relevance'], 1.0) * 0.4
    )
    metrics['quality_score'] = round(quality_score, 2)
    
    return metrics

# Test evaluation
test_cases = [
    {
        'query': 'What is OPEC demand forecast?',
        'answer': 'OPEC forecasts 2.25 million bpd growth [1].',
        'sources': [{'text': 'OPEC forecast 2.25 million barrels per day demand growth'}]
    },
    {
        'query': 'What is crude inventory?',
        'answer': 'Inventories fell significantly.',  # No citation!
        'sources': [{'text': 'crude inventories decreased 1.5 million barrels'}]
    }
]

print("RAG Evaluation Results:\n")
for i, test in enumerate(test_cases, 1):
    print(f"Test Case {i}:")
    eval_result = evaluate_rag(test['query'], test['answer'], test['sources'])
    print(json.dumps(eval_result, indent=2))
    print()

# Auto-graded test
result = evaluate_rag('test query', 'test answer [1]', [{'text': 'test source'}])
assert 'quality_score' in result, "Must return quality score"
assert 'cites_sources' in result, "Must check for citations"
print("✓ Exercise 4 passed!")

## Summary

Congratulations! You've mastered RAG workflows in Dataiku. Key takeaways:

1. **RAG = Retrieve + Generate** - Always ground LLM responses in retrieved context
2. **Prompt engineering matters** - Structure prompts for citations and accuracy
3. **Optimize retrieval** - Adaptive top-k, filtering, and relevance thresholds
4. **Citations are essential** - Trust, transparency, and hallucination detection
5. **Evaluate systematically** - Measure retrieval, generation, and citation quality

## Best Practices

- Start with top_k=3, adjust based on query complexity
- Always require source citations in prompts
- Validate citations to detect hallucinations
- Use metadata filtering for temporal/source relevance
- Monitor RAG quality metrics in production
- Keep Knowledge Banks updated

## Next Steps

- **Module 3**: Build custom RAG applications with Python recipes
- **Module 4**: Deploy RAG as production APIs with monitoring
- **Advanced**: Implement re-ranking, hybrid search, multi-hop RAG

In [ ]:
key_takeaways([
    "Core concept from this notebook demonstrated with working code",
    "Key parameters and their effects on results",
    "When to apply this technique vs alternatives"
])